# Types, functions and JSON

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-03-types-functions-and-json.ipynb)

A query is only as correct as the types it works with. This chapter covers the data types you will meet,
how Spark converts between them, the string and date functions used every day, and how to work with arrays,
structs and JSON. Each section ends with the mistake it is most likely to prevent.

It ends with practice questions and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW contacts AS
SELECT * FROM VALUES (1, 'ana@x.com'), (2, 'ben@y.org'), (3, 'cy@x.com')
AS contacts(id, email)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW shipments AS
SELECT * FROM VALUES
  (1, '{"city": "Pune", "zip": "411001"}'),
  (2, '{"city": "Delhi", "zip": "110001"}')
AS shipments(id, address_json)
""")

## 1. Data types

Every column and every expression has a type. The common ones are:

| Type | Holds | Example literal |
| --- | --- | --- |
| `INT`, `BIGINT` | whole numbers | `42`, `42L` |
| `DECIMAL(p, s)` | exact numbers with a fixed scale, such as money | `1.5` |
| `DOUBLE` | approximate floating-point numbers | `1.5D` |
| `STRING` | text | `'abc'` |
| `DATE`, `TIMESTAMP` | calendar dates, and dates with a time | `DATE '2023-01-05'` |
| `BOOLEAN` | true or false | `true` |
| `ARRAY`, `STRUCT`, `MAP` | collections and records | `array(1, 2)` |

`typeof` shows the type of an expression. Note that the literal `1.5` is a `DECIMAL`, not a `DOUBLE`.

In [ ]:
spark.sql("""
SELECT typeof(42)              AS int_literal,
       typeof(42L)             AS bigint_literal,
       typeof(1.5)             AS decimal_literal,
       typeof(1.5D)            AS double_literal,
       typeof('abc')           AS string_literal,
       typeof(DATE '2023-01-05') AS date_literal
""").show(truncate=False)

**Division is a common trap.** In Spark, `/` always returns a `DOUBLE`, so `10 / 4` is `2.5`. Use `DIV` when you
want integer division.

In [ ]:
spark.sql("""
SELECT 10 / 4   AS true_division,
       10 DIV 4 AS integer_division
""").show()

## 2. Casting and implicit conversion

`CAST(value AS type)` converts a value explicitly. Spark also converts types implicitly when it can, for example
when you add a string to a number. Spark 4 runs in ANSI mode by default, so a malformed cast raises an error
instead of returning NULL. Use `try_cast` when bad input should become NULL.

A cast that fails looks like this (not run here, because it stops the notebook):

```text
CAST('abc' AS INT)   -- CAST_INVALID_INPUT: the value 'abc' cannot be cast to INT
```

In [ ]:
spark.sql("""
SELECT try_cast('12'  AS INT) AS valid_text,
       try_cast('abc' AS INT) AS malformed_text
""").show()

spark.sql("SELECT '10' + 5 AS implicit_conversion").show()

**Dates stored as text.** `order_date = '2023-01-05'` works because Spark converts the string to a date. This is
a convenience, not a guarantee: a column of text dates sorts and compares as text, and a malformed value fails
only at query time. Store dates as `DATE` or `TIMESTAMP`.

## 3. String functions

| Function | Does |
| --- | --- |
| `concat(a, b)`, `concat_ws(sep, a, b)` | join strings; `concat_ws` skips the separator for NULLs |
| `upper`, `lower`, `trim` | change case; remove spaces at the ends |
| `substring(s, start, length)` | part of a string; `start` is 1-based |
| `length(s)` | number of characters |
| `replace(s, from, to)` | replace every occurrence |
| `split(s, sep)` | split into an array |
| `regexp_extract(s, pattern, group)` | the part of `s` matched by a regular expression |
| `LIKE`, `RLIKE` | simple pattern match, or a regular-expression match |

In [ ]:
spark.sql("""
SELECT concat_ws(' ', 'Acme', 'Corp')   AS joined,
       upper(trim('  pune  '))           AS cleaned,
       substring('abcdef', 2, 3)         AS middle,
       replace('2023-01-05', '-', '/')   AS slashed
""").show()

spark.sql("""
SELECT id,
       email,
       split(email, '@')[1]                       AS domain,
       regexp_extract(email, '^([a-z]+)', 1)     AS username
FROM contacts
ORDER BY id
""").show()

## 4. Dates and times

`DATE` holds a day. `TIMESTAMP` holds a day and a time. Date arithmetic is done with functions or with
`INTERVAL` literals:

| Need | Use |
| --- | --- |
| add or subtract days | `date_add(d, n)`, `date_sub(d, n)` |
| difference in days | `datediff(later, earlier)` |
| add months | `add_months(d, n)` |
| parts of a date | `year(d)`, `month(d)`, `extract(MONTH FROM d)` |
| start of a period | `date_trunc('month', ts)`, `trunc(d, 'MM')` |
| parse text | `to_date('07/05/2023', 'dd/MM/yyyy')` |
| format as text | `date_format(d, 'yyyy-MM')` |

Use a fixed reference date in examples, not `current_date()`. A query that depends on today's date returns a
different result every day, which makes it hard to check.

In [ ]:
spark.sql("""
SELECT order_id,
       order_date,
       datediff(DATE '2023-04-01', order_date) AS days_open,
       month(order_date)                       AS month_number
FROM orders
WHERE order_id IN (1, 14)
ORDER BY order_id
""").show()

spark.sql("""
SELECT add_months(DATE '2023-01-31', 1)            AS one_month_later,
       DATE '2023-01-31' + INTERVAL 1 MONTH        AS same_with_interval,
       TIMESTAMP '2023-01-01 10:00:00' - INTERVAL 30 MINUTES AS half_hour_earlier,
       date_trunc('month', TIMESTAMP '2023-05-07 10:20:00') AS month_start,
       to_date('07/05/2023', 'dd/MM/yyyy')        AS parsed
""").show(truncate=False)

## 5. Arrays, structs and maps

An **array** is an ordered list. A **struct** is a record with named fields. `explode` turns an array into one
row per element, and `collect_list` does the reverse, gathering rows into an array.

In [ ]:
spark.sql("""
SELECT size(array(1, 2, 3))            AS element_count,
       array_contains(array(1, 2), 2)  AS has_two,
       element_at(array('x', 'y'), 2)  AS second_element,
       named_struct('city', 'Pune', 'zip', '411001').city AS city_field
""").show()

spark.sql("SELECT explode(array(10, 20)) AS amount").show()

## 6. JSON

JSON often arrives as text. Spark can read a field with `get_json_object`, which returns a string, or parse the
whole document with `from_json` against a schema, which returns a typed struct. `to_json` goes the other way.

In [ ]:
spark.sql("""
SELECT id,
       get_json_object(address_json, '$.city')   AS city,
       from_json(address_json, 'city STRING, zip STRING').zip AS zip
FROM shipments
ORDER BY id
""").show()

spark.sql("SELECT to_json(named_struct('city', 'Pune')) AS as_json").show(truncate=False)

## Interview notes

- **Money.** Use `DECIMAL`, not `DOUBLE`. Binary floating point cannot represent 0.1 exactly. The next
  example adds the two values in both types:

In [ ]:
spark.sql("""
SELECT CAST(0.1 AS DOUBLE) + CAST(0.2 AS DOUBLE) AS as_double,
       0.1 + 0.2                                  AS as_decimal
""").show()

- **Integer division.** `/` returns a double in Spark, but some engines truncate `10 / 4` to `2`. Know which one
  your engine does before you compute an average or a ratio.
- **Implicit conversion.** When the types differ, the engine converts one side. Joining an `INT` column to a
  `STRING` column can silently stop matching rows or slow the query. Keep join keys the same type.

## Practice questions

1. How many orders were placed in March 2023? Count them.
2. For each contact, show the domain of the email (the part after `@`), sorted by `id`.
3. Extract the `city` of each shipment from its JSON address. Sort by `id`.
4. What does `0.1 + 0.2` return in Spark when both values are literals, and why is that different from doing the same sum with `DOUBLE`?

## Answers

### Question 1

In [ ]:
q = "SELECT COUNT(*) AS march_orders FROM orders WHERE month(order_date) = 3 AND year(order_date) = 2023"
spark.sql(q).show()
assert spark.sql(q).collect() == [(5,)]

### Question 2

In [ ]:
q = "SELECT id, split(email, '@')[1] AS domain FROM contacts ORDER BY id"
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, "x.com"), (2, "y.org"), (3, "x.com")]

### Question 3

In [ ]:
q = "SELECT id, get_json_object(address_json, '$.city') AS city FROM shipments ORDER BY id"
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, "Pune"), (2, "Delhi")]

### Question 4

Both literals are `DECIMAL`, so the sum is exact and equals `0.3`. Python's `float` and Spark's `DOUBLE` are
binary floating point, where `0.1 + 0.2` is `0.30000000000000004`. The next cell shows the double result.

In [ ]:
q = "SELECT CAST(0.1 AS DOUBLE) + CAST(0.2 AS DOUBLE) AS total"
spark.sql(q).show()
assert spark.sql(q).collect()[0][0] != 0.3